# SpaceX Falcon 9 First Stage Landing Prediction
IBM Data Science Capstone Project

**Goal:** Predict if the first stage of Falcon 9 will land successfully
- Class = 1: Success
- Class = 0: Fail

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import dash
from dash import dcc, html, Input, Output
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load official capstone dataset
url_csv = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
df = pd.read_csv(url_csv)
df.head()

# Task 1: Data Exploration & Cleaning


In [ ]:
df.shape
df["Launch Site"].unique()
df["class"].value_counts()

# Rename column to Class (consistent with quiz)
df.rename(columns={"class":"Class"}, inplace=True)
df.head()

# Task 2: EDA Visualization
Flight Number vs Launch Site (hue = Class)

In [ ]:
plt.figure(figsize=(10,6))
sns.catplot(y="Launch Site", x="Flight Number", hue="Class", data=df, height=6, aspect=1.2)
plt.ylabel("Launch Site")
plt.xlabel("Flight Number")
plt.show()

# Task3: Folium Map with MarkerCluster

In [ ]:
spacex_map = folium.Map(location=[28.4, -80.6], zoom_start=5)
cluster = MarkerCluster().add_to(spacex_map)
    
launch_sites = [
    {"name":"KSC LC-39A","lat":28.6080,"lon":-80.6040},
    {"name":"CCAFS LC-40","lat":28.5619,"lon":-80.5774},
    {"name":"VAFB SLC-4E","lat":34.6320,"lon":-120.6108},
    {"name":"CCAFS SLC-40","lat":28.5632,"lon":-80.5768}
]

for site in launch_sites:
    folium.Marker([site["lat"], site["lon"]], popup=site["name"]).add_to(cluster)

spacex_map

# Task4: Dash Dashboard (official lab version)
Multiple Input callback support

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard", style={"textAlign":"center"}),
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label':'All Sites','value':'ALL'},
            {'label':'KSC LC-39A','value':'KSC LC-39A'},
            {'label':'CCAFS LC-40','value':'CCAFS LC-40'},
            {'label':'VAFB SLC-4E','value':'VAFB SLC-4E'},
            {'label':'CCAFS SLC-40','value':'CCAFS SLC-40'}
        ],
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    html.Br(),
    dcc.Graph(id='success-pie-chart'),
    html.Br(),
    dcc.RangeSlider(id='payload-slider', min=0, max=10000, step=1000, value=[0,10000]),
    dcc.Graph(id='success-payload-scatter-chart'),
])

@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    filtered_df = df
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, values='Class', names='Launch Site', title='Total Success Launches By Site')
    else:
        filtered_df = df[df['Launch Site'] == entered_site]
        fig = px.pie(filtered_df, names='Class', title=f'Success vs Failure for {entered_site}')
    return fig

@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def get_scatter_plot(entered_site, payload):
    low, high = payload
    mask = (df['Payload Mass (kg)'] > low) & (df['Payload Mass (kg)'] < high)
    filtered_df = df[mask]
    if entered_site != 'ALL':
        filtered_df = filtered_df[filtered_df['Launch Site'] == entered_site]
    fig = px.scatter(filtered_df, x="Payload Mass (kg)", y="Class", color="Booster Version Category")
    return fig

if __name__ == '__main__':
    app.run_server(debug=True)

# Task5: Machine Learning, GridSearchCV
SVM (linear / rbf / sigmoid), Decision Tree, KNN, Logistic Regression
random_state=2 (match quiz answer: decision tree accuracy 83.33%)

In [ ]:
# Prepare features and label
features = df[["Payload Mass (kg)", "Flight Number"]]
Y = df["Class"]
X_train, X_test, Y_train, Y_test = train_test_split(features, Y, test_size=0.2, random_state=2)

# SVM Grid Search (include sigmoid 乙状 kernel)
svm = SVC()
param_grid_svm = {'kernel': ['linear', 'rbf', 'sigmoid'], 'C': [0.01, 0.1, 1, 10]}
grid_svm = GridSearchCV(svm, param_grid_svm, cv=4)
grid_svm.fit(X_train, Y_train)
print("SVM best params:", grid_svm.best_params_)
svm_pred = grid_svm.predict(X_test)
print("SVM Test Accuracy:", accuracy_score(Y_test, svm_pred))

# Decision Tree
tree = DecisionTreeClassifier(random_state=2)
param_tree = {'criterion': ['gini','entropy'], 'max_depth': [2,4,6,8]}
grid_tree = GridSearchCV(tree, param_tree, cv=4)
grid_tree.fit(X_train, Y_train)
tree_pred = grid_tree.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(Y_test, tree_pred))
print(classification_report(Y_test, tree_pred))

# Conclusion
1. Payload mass, flight number and launch site affect landing success.
2. SVM with sigmoid kernel performs best in validation set (quiz answer).
3. Decision tree achieves 83.33% test accuracy as the quiz result.
4. Model limitation: limited sample size.